In [ ]:
def __init__(self, props=mi.Properties()):
        super().__init__(props)
        self.max_cameradepth = int(props.get('max_cameradepth', 3))
        self.max_emitterdepth = int(props.get('max_emitterdepth', 3))
        self.light_image_buffer = None
        self.film_size = None
        self.sensor = None
        self.film = None
        self.size = None
        
    def prepare_buffer(self, scene):
        self.sensor: mi.SensorPtr = scene.sensors()[0]
        self.film = self.sensor.film()
        self.size = self.film.crop_size()
        self.film_size = mi.Vector2u(self.size.x, self.size.y)
        self.light_image_buffer = dr.zeros(mi.Float, self.size.x * self.size.y * 3)

    def sample(self,
               mode: dr.ADMode, 
               scene: mi.Scene,
               sampler: mi.Sampler,
               ray: mi.Ray3f,
               active: mi.Bool,
               **kwargs):
        
        ctx_c = mi.BSDFContext(mi.TransportMode.Radiance)
        ctx_e = mi.BSDFContext(mi.TransportMode.Importance) 
        
        L = mi.Spectrum(0.0)
        
        EPW: list[mi.Spectrum] = [mi.Spectrum(0.0) for _ in range(self.max_emitterdepth + 1)]
        EPS: list[mi.SurfaceInteraction3f] = [dr.zeros(mi.SurfaceInteraction3f) for _ in range(self.max_emitterdepth + 1)]
        EPA = [mi.Bool(False) for _ in range(self.max_emitterdepth + 1)]
        EPM = [dr.zeros(mi.Float) for _ in range(self.max_emitterdepth + 1)]
        EP_PDF = [dr.zeros(mi.Float) for _ in range(self.max_emitterdepth + 1)]
        
        emitter_weight = dr.zeros(mi.Spectrum)
        si = dr.zeros(mi.SurfaceInteraction3f)
        
        emitter_idx, emitter_idx_weight, _ = scene.sample_emitter(sampler.next_1d(), active)
        emitter = dr.gather(mi.EmitterPtr, scene.emitters_dr(), emitter_idx)
        
        is_infinite = mi.has_flag(emitter.flags(), mi.EmitterFlags.Infinite)
        is_delta = mi.has_flag(emitter.flags(), mi.EmitterFlags.Delta)
        active_e = active & ~is_infinite & ~is_delta
        
        ps, pos_weight = emitter.sample_position(0.0, sampler.next_2d(), active_e)
        emitter_weight[active_e] = pos_weight * emitter_idx_weight
        
        si.p[active_e] = ps.p
        si.n[active_e] = ps.n
        si.uv[active_e] = ps.uv
        si.time[active_e] = ps.time
        si.sh_frame[active_e] = mi.Frame3f(ps.n)
        
        wo_local = mi.warp.square_to_cosine_hemisphere(sampler.next_2d())
        wavelenghts, wavelenghts_weight = emitter.sample_wavelengths(si, sampler.next_1d(), active_e)
        si.wavelengths = wavelenghts
        
        si.wi[active_e] = wo_local
        wo_world = si.to_world(wo_local)
        
        EPW[0] = emitter_weight
        EPS[0] = si
        EPA[0] = active_e
        EPM[0] = 0.0
        
        pdf_emit = ps.pdf / emitter_idx_weight # 광원 표면 면적 확률
        EP_PDF[0] = pdf_emit
        emitter_ray = si.spawn_ray(wo_world)
        
        pdf_dir = mi.warp.square_to_cosine_hemisphere_pdf(wo_local)
        safe_pdf_dir = dr.maximum(1e-6, pdf_dir)
        
        path_weight = emitter_weight * wavelenghts_weight * dr.pi 
        path_weight = dr.select(active_e, path_weight, 0.0)
        
        prev_pdf_fwd = safe_pdf_dir * pdf_emit
        prev_n = si.n
        
        # emitter path 생성
        for i in range(self.max_emitterdepth):
            si = scene.ray_intersect(emitter_ray)
            active_e &= si.is_valid()
            
            bsdf_sample, bsdf_weight = si.bsdf().sample(ctx_e, si, sampler.next_1d(), sampler.next_2d(), active_e)
            
            pdf_fwd_next = bsdf_sample.pdf
            
            si_rev = dr.zeros(mi.SurfaceInteraction3f)
            si_rev[active_e] = si
            si_rev.wi = bsdf_sample.wo 
            wo_rev = si.wi            
            _, pdf_rev = si.bsdf().eval_pdf(ctx_e, si_rev, wo_rev, active_e)
            
            cos_curr = dr.abs(dr.dot(si.n, -emitter_ray.d))
            area_pdf_e = dr.select(si.t > 1e-6, prev_pdf_fwd * cos_curr / (si.t * si.t), 0.0)
            
            cos_prev = dr.abs(dr.dot(prev_n, emitter_ray.d))
            
            prev_pdf_fwd = prev_pdf_fwd * cos_curr
            pdf_rev = pdf_rev * cos_prev
            
            mis_ratio = dr.select(prev_pdf_fwd > 1e-6, pdf_rev / prev_pdf_fwd, 0.0)
            mis_ratio = mis_ratio * mis_ratio
            
            idx = i + 1
            EPM[idx] = mis_ratio * (1.0 + EPM[i])
            EPS[idx] = si
            EPW[idx] = path_weight
            EPA[idx] = active_e
            EP_PDF[idx] = area_pdf_e
            
            path_weight = path_weight * bsdf_weight
            prev_pdf_fwd = pdf_fwd_next
            prev_n = si.n
            emitter_ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
        
        # 센서에 직접 연결
        for j in range(0, self.max_emitterdepth + 1): 
            si_e = EPS[j]
            weight_e = EPW[j]
            active_e_node = EPA[j]
            mis_light = EPM[j]
            
            ds, sensor_val = self.sensor.sample_direction(si_e, sampler.next_2d(), active_e_node)
            active_conn = active_e_node & (ds.pdf > 0.0)
            
            shadow_ray = si_e.spawn_ray_to(ds.p)
            active_conn &= ~scene.ray_test(shadow_ray)
            
            d_vec = ds.d
            dist_sq = dr.maximum(ds.dist * ds.dist, 1e-6)
            cos_e = dr.abs(dr.dot(si_e.n, d_vec))
            cos_c = dr.abs(dr.dot(ds.n, -d_vec))
            
            pdf_sensor_solid = ds.pdf
            pdf_cam_area = dr.select(dist_sq > 1e-6, pdf_sensor_solid * cos_e / dist_sq, 0.0)
            
            wo_local = si_e.to_local(d_vec)
            _, pdf_light_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, wo_local, active_conn)
            pdf_light_area = dr.select(dist_sq > 1e-6, pdf_light_solid * cos_c / dist_sq, 0.0)
            
            ratio_cam_side = dr.select(pdf_light_area > 1e-6, pdf_cam_area / pdf_light_area, 0.0)
            ratio_cam_side *= ratio_cam_side
            
            w_cam_side = dr.select(pdf_light_area > 1e-6, ratio_cam_side * (1.0 + mis_light), 0.0)
            
            mis_weight = 1.0 / (1.0 + w_cam_side)
            mis_weight = dr.select(active_conn & dr.isfinite(mis_weight), mis_weight, 0.0)
            mis_weight = 1.0
            
            bsdf_val = si_e.bsdf().eval(ctx_e, si_e, wo_local, active_conn)
            L_connect = weight_e * sensor_val * bsdf_val
            L_connect = dr.select(dr.isfinite(L_connect), L_connect, 0.0)
            L_connect = dr.clip(L_connect, 0.0, 1000.0)
            L_connect = dr.select(active_conn, L_connect * mis_weight, 0.0)
            
            width_f = mi.Float(self.film_size.x)
            height_f = mi.Float(self.film_size.y)
            
            pos_x = dr.clip(dr.floor(ds.uv.x), 0.0, width_f - 1.0)
            pos_y = dr.clip(dr.floor(ds.uv.y), 0.0, height_f - 1.0)
            pixel_idx = (mi.UInt32(pos_y) * self.film_size.x + mi.UInt32(pos_x)) * 3
            
            dr.scatter_reduce(dr.ReduceOp.Add, target=self.light_image_buffer, value=L_connect[0], index=pixel_idx + 0, active=active_conn)
            dr.scatter_reduce(dr.ReduceOp.Add, target=self.light_image_buffer, value=L_connect[1], index=pixel_idx + 1, active=active_conn)
            dr.scatter_reduce(dr.ReduceOp.Add, target=self.light_image_buffer, value=L_connect[2], index=pixel_idx + 2, active=active_conn)
        

        beta_c = mi.Spectrum(1.0)
        active_c = active
        mis_cam = mi.Float(0.0)
        
        prev_pdf_fwd_c = mi.Float(1.0) 
        prev_n_c = dr.zeros(mi.Vector3f)
        prev_si = dr.zeros(mi.SurfaceInteraction3f)
        prev_bsdf_delta = mi.Bool(True)
        
        si = scene.ray_intersect(ray)
        
        # 센서 path 생성 및 연결
        for i in range(self.max_cameradepth):
            active_c &= si.is_valid()
            bsdf = si.bsdf()
            
            cos_curr_c = dr.abs(dr.dot(si.n, -ray.d))
            pdf_fwd_A_c = dr.select(si.t > 1e-6, prev_pdf_fwd_c * cos_curr_c / (si.t * si.t), 0.0)
            
            hit_emitter_ptr = si.emitter(scene)
            active_hit = active_c
                
            ds = mi.DirectionSample3f(scene, si=si, ref=prev_si)
            pdf_nee = scene.pdf_emitter_direction(prev_si, ds, ~prev_bsdf_delta)
                
            w_nee = dr.select(prev_pdf_fwd_c > 1e-6, pdf_nee / prev_pdf_fwd_c, 0.0)
            w_nee = w_nee * w_nee
                
            denom_w = 1.0 + w_nee * (1.0 + mis_cam) 
                
            mis_weight_bsdf = dr.select(denom_w > 1e-6, 1.0 / denom_w, 0.0)
            mis_weight_bsdf = dr.select(dr.isfinite(mis_weight_bsdf), mis_weight_bsdf, 0.0)
                
            Le_bsdf_hit = hit_emitter_ptr.eval(si, active_hit)
            Le_bsdf_hit = dr.clip(Le_bsdf_hit, 0.0, 1000.0)
            
            L += dr.select(active_hit, beta_c * Le_bsdf_hit * mis_weight_bsdf, 0.0)
            
            for j in range(0, self.max_emitterdepth + 1): 
                si_e = EPS[j]
                weight_e = EPW[j]
                active_e_node = EPA[j]
                
                active_conn = active_c & active_e_node
                d_conn = si_e.p - si.p
                dist_sq = dr.maximum(dr.squared_norm(d_conn), 1e-6)
                dist = dr.sqrt(dist_sq)
                wo_c = d_conn / dist
                wo_e = -wo_c
                
                shadow_ray = si.spawn_ray_to(si_e.p)
                is_delta_cam = mi.has_flag(bsdf.flags(), mi.BSDFFlags.Delta)
                is_delta_emit = mi.Bool(False)
                if j > 0:
                    is_delta_emit = mi.has_flag(si_e.bsdf().flags(), mi.BSDFFlags.Delta)
                    
                visible = active_conn & ~scene.ray_test(shadow_ray) & ~is_delta_cam & ~is_delta_emit
                
                cos_c = dr.abs(dr.dot(si.n, wo_c))
                cos_e = dr.abs(dr.dot(si_e.n, wo_e))
                
                _, pdf_c2l = bsdf.eval_pdf(ctx_c, si, si.to_local(wo_c), visible)
                p_A_c2l = pdf_c2l * cos_e / dist_sq
                
                p_A_l2c = mi.Float(0.0)
                if j == 0:
                    p_A_l2c = pdf_emit 
                else:
                    _, pdf_l2c_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    p_A_l2c = pdf_l2c_solid * cos_c / dist_sq
                
                # 🚨 [MAJOR MATHEMATICAL FIX] 연결 확률을 생성 면적 확률로 나눕니다!
                pdf_fwd_A_e = EP_PDF[j]
                
                # 1. 카메라 방향으로 Vertex를 넘길 때의 비율 (Camera가 si_e를 생성할 확률 / Light가 생성했던 확률)
                ratio_towards_camera = dr.select(pdf_fwd_A_e > 1e-6, p_A_c2l / pdf_fwd_A_e, 0.0)
                ratio_towards_camera = ratio_towards_camera * ratio_towards_camera # [DrJit Safe, Power Heuristic]
                
                # 2. 광원 방향으로 Vertex를 넘길 때의 비율 (Light가 si를 생성할 확률 / Camera가 생성했던 확률)
                if i == 0:
                    # 💡 i=0 (t=2)일 때, 카메라는 핀홀(Delta)이므로 확률이 무한대. 따라서 비율은 0이 맞습니다!
                    ratio_towards_light = mi.Float(0.0)
                else:
                    ratio_towards_light = dr.select(pdf_fwd_A_c > 1e-6, p_A_l2c / pdf_fwd_A_c, 0.0)
                    ratio_towards_light = ratio_towards_light * ratio_towards_light # [DrJit Safe, Power Heuristic]
                
                w_light_side = ratio_towards_light * (1.0 + EPM[j])
                w_cam_side = ratio_towards_camera * (1.0 + mis_cam)
                
                mis_weight = 1.0 / (1.0 + w_cam_side + w_light_side)
                mis_weight = dr.select(visible & dr.isfinite(mis_weight), mis_weight, 0.0)
                
                f_cam = bsdf.eval(ctx_c, si, si.to_local(wo_c), visible)
                contrib = mi.Spectrum(0.0)
                
                if j == 0:
                    si_e.wi = si_e.to_local(wo_e)
                    Le_connect = emitter.eval(si_e, visible)
                    G = cos_e / dist_sq
                    contrib = beta_c * Le_connect * weight_e * G * f_cam
                else:
                    f_emit = si_e.bsdf().eval(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    G = 1.0 / dist_sq
                    contrib = beta_c * f_cam * G * f_emit * weight_e 
                
                contrib = dr.select(dr.isfinite(contrib), contrib, 0.0) # [NaN 방어 추가]
                L += dr.select(visible, contrib * mis_weight, 0.0)
            
            
            bsdf_sample, bsdf_weight = bsdf.sample(ctx_c, si, sampler.next_1d(), sampler.next_2d(), active_c)
            
            si_rev_c = dr.zeros(mi.SurfaceInteraction3f)
            si_rev_c[active_c] = si
            si_rev_c.wi = bsdf_sample.wo 
            wo_rev_c = si.wi             
            _, pdf_rev_c = si.bsdf().eval_pdf(ctx_c, si_rev_c, wo_rev_c, active_c)
            
            cos_curr_c = dr.abs(dr.dot(si.n, -ray.d))
            cos_prev_c = dr.abs(dr.dot(prev_n_c, ray.d))
            
            denom_c = prev_pdf_fwd_c * cos_curr_c
            numer_c = pdf_rev_c * cos_prev_c
            
            ratio = dr.select(denom_c > 1e-6, numer_c / denom_c, 0.0)
            ratio = ratio * ratio
            
            mis_cam = ratio * (1.0 + mis_cam) 
            
            prev_pdf_fwd_c = bsdf_sample.pdf
            prev_n_c = si.n
            prev_si = si
            beta_c *= bsdf_weight
            prev_bsdf_delta = mi.has_flag(bsdf_sample.sampled_type, mi.BSDFFlags.Delta)
            
            ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
            
        dr.eval(self.light_image_buffer)
        
        return L, active, [], None

In [ ]:
class PathIntegratorFinal(mi.ad.integrators.common.ADIntegrator):
    def __init__(self, props = mi.Properties()):
        super().__init__(props)
        self.max_caemradepth = int(props.get('max_cameradepth', 2))
        self.max_emitterdepth = int(props.get('max_emitterdepth', 2))
        self.light_image_buffer = None
        self.film_size = None
        self.sensor = None
        self.film = None
        self.size = None
        
    def prepare_buffer(self, scene):
        self.sensor = scene.sensors()[0]
        self.film = self.sensor.film()
        self.size = self.film.crop_size()
        self.film_size = mi.Vector2u(self.size.x, self.size.y)
        self.light_image_buffer = dr.zeros(mi.Float, self.size.x * self.size.y * 3)

    def sample(self,
               mode: dr.ADMode, # NOT USED NOW (differentiable rendering)
               scene: mi.Scene,
               sampler: mi.Sampler,
               ray: mi.Ray3f,
               active: mi.Bool,
               **kwargs # Absorbs unused arguments
               ):
        
        ctx_c = mi.BSDFContext(mi.TransportMode.Radiance)
        ctx_e = mi.BSDFContext(mi.TransportMode.Importance) 
        
        L = mi.Spectrum(0.0)
        
        EPW : list[mi.Spectrum] = [mi.Spectrum(0.0) for _ in range(self.max_emitterdepth + 1)]
        EPS: list[mi.SurfaceInteraction3f] = [dr.zeros(mi.SurfaceInteraction3f) for _ in range(self.max_emitterdepth + 1)]
        EPA = [mi.Bool(False) for _ in range(self.max_emitterdepth + 1)]
        EPM = [dr.zeros(mi.Float) for _ in range(self.max_emitterdepth + 1)]
        
        emitter_weight = dr.zeros(mi.Spectrum)
        si = dr.zeros(mi.SurfaceInteraction3f)
        
        emitter_idx, emitter_idx_weight, _ = scene.sample_emitter(sampler.next_1d(), active)
        emitter = dr.gather(mi.EmitterPtr, scene.emitters_dr(), emitter_idx)
        
        is_infinite = mi.has_flag(emitter.flags(), mi.EmitterFlags.Infinite)
        is_delta = mi.has_flag(emitter.flags(), mi.EmitterFlags.Delta)
        
        active_e = active & ~is_infinite & ~is_delta
        ps, pos_weight = emitter.sample_position(0.0, sampler.next_2d(), active_e)
        
        emitter_weight[active_e] = pos_weight*emitter_idx_weight
        
        si.p[active_e] = ps.p
        si.n[active_e] = ps.n
        si.uv[active_e] = ps.uv
        si.time[active_e] = ps.time
        si.sh_frame[active_e] = mi.Frame3f(ps.n)
        
        wo_local = mi.warp.square_to_cosine_hemisphere(sampler.next_2d())
        
        
        wavelenghts, wavelenghts_weight = emitter.sample_wavelengths(si, sampler.next_1d(), active_e)
        si.wavelengths = wavelenghts
        
        si.wi[active_e] = wo_local
        wo_world = si.to_world(wo_local)
        
        EPW[0] = emitter_weight
        EPS[0] = si
        EPA[0] = active_e
        EPM[0] = 0.0
        
        pdf_emit = ps.pdf/emitter_idx_weight
        
        emitter_ray = si.spawn_ray(wo_world)
        
        
        pdf_dir = mi.warp.square_to_cosine_hemisphere_pdf(wo_local)
        safe_pdf_dir = dr.maximum(1e-6, pdf_dir)
        
        Le = emitter.eval(si, active_e)
        
        path_weight = emitter_weight * dr.pi * wavelenghts_weight #* Le 
        path_weight = dr.select(active_e, path_weight, 0.0)
        
        prev_pdf_fwd = safe_pdf_dir*pdf_emit
        prev_n = si.n
        
        for i in range(self.max_emitterdepth):
            
            si = scene.ray_intersect(emitter_ray)
            active_e &= si.is_valid()
            
            _, pdf_rev = si.bsdf().eval_pdf(ctx_e, si, si.to_local(-emitter_ray.d), active_e)
            
            cos_curr = dr.maximum(0.0, dr.dot(si.n, -emitter_ray.d))
            cos_prev = dr.maximum(0.0, dr.dot(prev_n, emitter_ray.d))
            
            denom = prev_pdf_fwd * cos_curr
            numer = pdf_rev * cos_prev
            
            mis_ratio = dr.select(denom > 1e-6, numer / denom, 0.0)
            #mis_ratio *= mis_ratio
            
            idx = i + 1
            
            EPM[idx] = mis_ratio * (1.0 + EPM[i])
            EPS[idx] = si
            EPW[idx] = path_weight
            EPA[idx] = active_e
            
            bsdf_sample, bsdf_weight = si.bsdf().sample(ctx_e, si, sampler.next_1d(), sampler.next_2d(), active_e)
            path_weight *= bsdf_weight
            
            prev_pdf_fwd = bsdf_sample.pdf
            prev_n = si.sh_frame.n
            emitter_ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
            
        for j in range(1, self.max_emitterdepth+1): # connect c0 - li - .. - l0
            si_e = EPS[j]
            weight_e = EPW[j]
            active_e_node = EPA[j]
            mis_light = EPM[j]
            
            is_delta_emit = mi.Bool(False)
            if j > 0:
                is_delta_emit = mi.has_flag(si_e.bsdf().flags(), mi.BSDFFlags.Delta)
                
            ds, sensor_val = self.sensor.sample_direction(si_e, sampler.next_2d(), active_e_node)
            active_conn = active_e_node & ~is_delta_emit
             
            shadow_ray = si_e.spawn_ray_to(ds.p)
            active_conn &= ~scene.ray_test(shadow_ray)
            
            d_vec = ds.d
            dist_sq = ds.dist * ds.dist
            
            cos_e = dr.abs(dr.dot(si_e.n, d_vec))
            cos_c = dr.abs(dr.dot(ds.n, -d_vec))
            
            #active_conn &= (cos_e > 0.0) & (cos_c > 0.0)
            
            pdf_sensor_solid = ds.pdf
            pdf_cam_area = dr.select(dist_sq > 1e-6, pdf_sensor_solid * dr.abs(cos_e) / dist_sq, 0.0)
            
            #p_l2c = dr.select(dist_sq > 1e-6, ds.pdf * dr.abs(cos_c) / dist_sq, 0.0)
            
            pdf_light_solid = mi.Float(1.0)
            pdf_light_area = mi.Float(1.0)
            if j == 0:
                #   Emitter 자체인 경우 (Cosine 분포)
                wo_local = si_e.to_local(d_vec)
                pdf_light_solid = mi.warp.square_to_cosine_hemisphere_pdf(wo_local)
                pdf_light_area = dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
                pdf_light_area *= pdf_emit #dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
            else:
                # 표면 반사인 경우 (BSDF 확률)
                wo_local = si_e.to_local(d_vec)
                _, pdf_light_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, wo_local, active_conn)
                pdf_light_area = dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
            
            connectpdf = (pdf_cam_area / pdf_light_area)
            #connectpdf *= connectpdf
            
            ratio_cam_vs_light = dr.select(pdf_light_area > 1e-6, mis_light * connectpdf, 0.0)
            
            mis_weight = 1.0 / (1.0 + ratio_cam_vs_light)
            mis_weight = dr.select(active_conn, mis_weight, 0.0)
            mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                     
            L_connect = mi.Spectrum(0.0)
            
            
            if(j == 0):
                si_e.wi = si_e.to_local(wo_local)
                Le = emitter.eval(si_e, active_conn)
                L_connect = weight_e * sensor_val * Le * cos_e
            else:
                G = 1.0 / dist_sq
                bsdf_val = si_e.bsdf().eval(ctx_e, si_e, wo_local, active_conn)
                L_connect = weight_e  * sensor_val * bsdf_val#* bsdf_val
                
            L_connect = dr.select(active_conn, L_connect * mis_weight, 0.0)
            
            
            width_f = mi.Float(self.film_size.x)
            height_f = mi.Float(self.film_size.y)
            
            pos_x = dr.clip(dr.floor(ds.uv.x), 0.0, width_f - 1.0)
            pos_y = dr.clip(dr.floor(ds.uv.y), 0.0, height_f - 1.0)
                
            pixel_idx = (mi.UInt32(pos_y) * self.film_size.x + mi.UInt32(pos_x)) * 3
            

            dr.scatter_add(target=self.light_image_buffer, value=L_connect[0], index=pixel_idx + 0, active=active_conn)
            dr.scatter_add(target=self.light_image_buffer, value=L_connect[1], index=pixel_idx + 1, active=active_conn)
            dr.scatter_add(target=self.light_image_buffer, value=L_connect[2], index=pixel_idx + 2, active=active_conn)
            
        
        beta_c = mi.Spectrum(1.0)
        active_c = active
        mis_cam = mi.Float(0.0)
        prev_pdf_fwd_c = mi.Float(0.0)
        prev_n_c = dr.zeros(mi.Vector3f)
        prev_si = dr.zeros(mi.SurfaceInteraction3f)
        
        prev_bsdf_delta =  mi.Bool(True)
        
        si = scene.ray_intersect(ray)
        Le_direct = si.emitter(scene).eval(si)
        L += beta_c * Le_direct # connect c0 - c1
        
        prev_pdf_fwd_c = mi.Float(1.0) #mi.warp.square_to_cosine_hemisphere_pdf(-si.wi)
        
        emitters = scene.emitters()
        emitters_len = len(emitters)
        
        for i in range(self.max_caemradepth):
            si = scene.ray_intersect(ray)
            active_c &= si.is_valid()
            
            bsdf = si.bsdf()
            
            if i > 0:
                ds = mi.DirectionSample3f(scene, si=si, ref=prev_si)
                pdf_nee = scene.pdf_emitter_direction(prev_si, ds, ~prev_bsdf_delta)
                
                hit_emitter_ptr = si.emitter(scene)
                
                pdf_emit_sel = dr.zeros(mi.Float, dr.width(si))
                
                for k in range(emitters_len):
                    e_ptr = emitters[k]
                    prob_k = scene.pdf_emitter(k)
                    is_match = (hit_emitter_ptr == e_ptr)
                    pdf_emit_sel = dr.select(is_match, prob_k, pdf_emit_sel)
                    
                ps_hit = dr.zeros(mi.PositionSample3f)
                ps_hit.p = si.p
                ps_hit.n = si.n
                ps_hit.uv = si.uv
                ps_hit.time = si.time
                
                pdf_emit_pos = si.shape.pdf_position(ps_hit, active_c)
                
                p_light_start_area = pdf_emit_sel * pdf_emit_pos
                
                dist_sq = dr.squared_norm(si.p - prev_si.p)
                cos_at_light = dr.maximum(0.0, dr.dot(si.n, -ray.d))
                p_light_solid = p_light_start_area * dist_sq / dr.maximum(1e-6, cos_at_light)
                
                w_nee = dr.select(prev_pdf_fwd_c > 1e-6, pdf_nee / prev_pdf_fwd_c, 0.0)
                w_light = dr.select(prev_pdf_fwd_c > 1e-6, mis_cam * (p_light_solid / prev_pdf_fwd_c), 0.0)
                
                denom_w = 1.0 + w_nee + w_light
                mis_weight_bsdf = dr.select(denom_w > 1e-6, 1 / denom_w, 0.0)
                mis_weight_bsdf = dr.select(dr.isfinite(mis_weight_bsdf), mis_weight_bsdf, 0.0)
                
                Le_bsdf_hit = hit_emitter_ptr.eval(si, active_c)
                L += beta_c * Le_bsdf_hit * mis_weight_bsdf # connect c0 - c1 - c(i+1)
            
            cos_curr = dr.maximum(0.0, dr.dot(si.sh_frame.n, -ray.d))
            cos_prev = dr.maximum(0.0, dr.dot(prev_n_c, ray.d))
            _, pdf_rev_c = si.bsdf().eval_pdf(ctx_c, si, si.to_local(-ray.d), ~prev_bsdf_delta)
                
                
            denom_c = prev_pdf_fwd_c * cos_curr
            numer_c = pdf_rev_c * cos_prev
            ratio = dr.select(denom_c > 1e-6, numer_c / denom_c, 0.0)
            #ratio *= ratio   
            mis_cam = ratio * (1.0 + mis_cam)
            
            for j in range(self.max_emitterdepth + 1):
                si_e = EPS[j]
                weight_e = EPW[j]
                active_e_node = EPA[j]
                
                active_conn = active_c & active_e_node
                
                d_conn = si_e.p - si.p
                dist_sq = dr.squared_norm(d_conn)
                dist = dr.sqrt(dist_sq)
                wo_c = d_conn / dist
                wo_e = -wo_c
                
                shadow_ray = si.spawn_ray_to(si_e.p)
                
                is_delta_cam = mi.has_flag(bsdf.flags(), mi.BSDFFlags.Delta)
                is_delta_emit = mi.Bool(False)
                if j > 0:
                    is_delta_emit = mi.has_flag(si_e.bsdf().flags(), mi.BSDFFlags.Delta)
                
                visible = active_conn & ~scene.ray_test(shadow_ray) & ~is_delta_cam & ~is_delta_emit
                
                cos_c = dr.dot(si.n, wo_c)
                cos_e = dr.dot(si_e.n, wo_e)
                
                _, pdf_c2l = bsdf.eval_pdf(ctx_c, si, si.to_local(wo_c), visible)
                p_A_c2l = pdf_c2l * dr.abs(cos_e) / dist_sq
                
                p_A_l2c = mi.Float(0.0)
                
                if j == 0:
                    p_A_l2c = pdf_emit #dr.select(weight_e > 1e-6, 1.0 / weight_e, 0.0)
                    pdf_dir = mi.warp.square_to_cosine_hemisphere_pdf(wo_e) * dr.abs(cos_c) / dist_sq
                    p_A_l2c *= pdf_dir
                else:
                    _, pdf_l2c_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    p_A_l2c = pdf_l2c_solid * dr.abs(cos_c) / dist_sq

                
                cam_side = p_A_c2l / p_A_l2c
                #cam_side *= cam_side
                
                light_side = p_A_l2c / p_A_c2l
                #light_side *= light_side
                
                w_cam_side = dr.select(p_A_l2c > 1e-6, (mis_cam) * (cam_side), 0.0)
                w_light_side = dr.select(p_A_c2l > 1e-6, EPM[j] * (light_side), 0.0)
                
                mis_weight = 1.0 / (1.0 + w_cam_side + w_light_side)
                mis_weight = dr.select(visible, mis_weight, 0.0)
                mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                
                
                contrib = mi.Spectrum(0.0)
                
                if j == 0:
                    si_e.wi = si_e.to_local(wo_e)
                    Le_connect = emitter.eval(si_e, visible)
                    f_cam = bsdf.eval(ctx_c, si, si.to_local(wo_c), visible)
                    G = cos_e / dist_sq
                    contrib = beta_c  * Le_connect * weight_e * G * f_cam
                    
                else:
                    f_cam = bsdf.eval(ctx_c, si, si.to_local(wo_c), visible)
                    f_emit = si_e.bsdf().eval(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    G = 1.0 / dist_sq
                    contrib = beta_c * f_cam * G * f_emit * weight_e 

                mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                L += dr.select(visible, contrib*mis_weight, 0.0) # connect c0 - c1 - c(i+1) - l(j) - l(0)


            bsdf_sample, bsdf_weight = bsdf.sample(ctx_c, si, sampler.next_1d(), sampler.next_2d(), active_c)
            prev_pdf_fwd_c = bsdf_sample.pdf
            prev_n_c = si.sh_frame.n
            prev_si = si
            beta_c *= bsdf_weight
            prev_bsdf_delta = mi.has_flag(bsdf_sample.sampled_type, mi.BSDFFlags.Delta)
            ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
            
        
        
        return L, active, [], None

In [ ]:
class PathIntegratorFinal(mi.ad.integrators.common.ADIntegrator):
    def __init__(self, props = mi.Properties()):
        super().__init__(props)
        self.max_caemradepth = int(props.get('max_cameradepth', 2))
        self.max_emitterdepth = int(props.get('max_emitterdepth', 2))
        self.light_image_buffer = None
        self.film_size = None
        self.sensor = None
        self.film = None
        self.size = None
        
    def prepare_buffer(self, scene):
        self.sensor : mi.SensorPtr = scene.sensors()[0]
        self.film = self.sensor.film()
        self.size = self.film.crop_size()
        self.film_size = mi.Vector2u(self.size.x, self.size.y)
        self.light_image_buffer = dr.zeros(mi.Float, self.size.x * self.size.y * 3)

    def sample(self,
               mode: dr.ADMode, # NOT USED NOW (differentiable rendering)
               scene: mi.Scene,
               sampler: mi.Sampler,
               ray: mi.Ray3f,
               active: mi.Bool,
               **kwargs # Absorbs unused arguments
               ):
        
        ctx_c = mi.BSDFContext(mi.TransportMode.Radiance)
        ctx_e = mi.BSDFContext(mi.TransportMode.Importance) 
        
        L = mi.Spectrum(0.0)
        
        EPW : list[mi.Spectrum] = [mi.Spectrum(0.0) for _ in range(self.max_emitterdepth + 1)]
        EPS: list[mi.SurfaceInteraction3f] = [dr.zeros(mi.SurfaceInteraction3f) for _ in range(self.max_emitterdepth + 1)]
        EPA = [mi.Bool(False) for _ in range(self.max_emitterdepth + 1)]
        EPM = [dr.zeros(mi.Float) for _ in range(self.max_emitterdepth + 1)]
        
        emitter_weight = dr.zeros(mi.Spectrum)
        si = dr.zeros(mi.SurfaceInteraction3f)
        
        emitter_idx, emitter_idx_weight, _ = scene.sample_emitter(sampler.next_1d(), active)
        emitter = dr.gather(mi.EmitterPtr, scene.emitters_dr(), emitter_idx)
        
        is_infinite = mi.has_flag(emitter.flags(), mi.EmitterFlags.Infinite)
        is_delta = mi.has_flag(emitter.flags(), mi.EmitterFlags.Delta)
        
        active_e = active & ~is_infinite & ~is_delta
        ps, pos_weight = emitter.sample_position(0.0, sampler.next_2d(), active_e)
        
        emitter_weight[active_e] = pos_weight*emitter_idx_weight
        
        si.p[active_e] = ps.p
        si.n[active_e] = ps.n
        si.uv[active_e] = ps.uv
        si.time[active_e] = ps.time
        si.sh_frame[active_e] = mi.Frame3f(ps.n)
        
        wo_local = mi.warp.square_to_cosine_hemisphere(sampler.next_2d())
        
        
        wavelenghts, wavelenghts_weight = emitter.sample_wavelengths(si, sampler.next_1d(), active_e)
        si.wavelengths = wavelenghts
        
        si.wi[active_e] = wo_local
        wo_world = si.to_world(wo_local)
        
        EPW[0] = emitter_weight
        EPS[0] = si
        EPA[0] = active_e
        EPM[0] = 0.0
        
        pdf_emit = ps.pdf/emitter_idx_weight
        
        emitter_ray = si.spawn_ray(wo_world)
        
        
        pdf_dir = mi.warp.square_to_cosine_hemisphere_pdf(wo_local)
        safe_pdf_dir = dr.maximum(1e-6, pdf_dir)
        
        Le = emitter.eval(si, active_e)
        
        path_weight = emitter_weight * dr.pi * wavelenghts_weight #* Le 
        path_weight = dr.select(active_e, path_weight, 0.0)
        
        si_e = scene.ray_intersect(emitter_ray)
        d_conn =  si.p - si_e.p
        dist_sq = dr.squared_norm(d_conn)
        dist = dr.sqrt(dist_sq)
        wo_c = d_conn / dist
        
        cos_c = dr.dot(si_e.n, wo_c)
        
        emitter_weight_soild = (pdf_emit)*dist_sq/cos_c
        
        prev_pdf_fwd = safe_pdf_dir*pdf_emit
        prev_n = si.n
        
        for i in range(self.max_emitterdepth):
            
            si = scene.ray_intersect(emitter_ray)
            active_e &= si.is_valid()
            
            _, pdf_rev = si.bsdf().eval_pdf(ctx_e, si, si.to_local(-emitter_ray.d), active_e)
            
            cos_curr = dr.maximum(0.0, dr.dot(si.n, -emitter_ray.d))
            cos_prev = dr.maximum(0.0, dr.dot(prev_n, emitter_ray.d))
            
            denom = prev_pdf_fwd * cos_curr
            numer = pdf_rev * cos_prev
            
            mis_ratio = dr.select(denom > 1e-6, numer / denom, 0.0)
            #mis_ratio *= mis_ratio
            
            idx = i + 1
            
            EPM[idx] = mis_ratio * (1.0 + EPM[i])
            EPS[idx] = si
            EPW[idx] = path_weight
            EPA[idx] = active_e
            
            bsdf_sample, bsdf_weight = si.bsdf().sample(ctx_e, si, sampler.next_1d(), sampler.next_2d(), active_e)
            path_weight *= bsdf_weight
            
            prev_pdf_fwd = bsdf_sample.pdf
            prev_n = si.sh_frame.n
            emitter_ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
            
        for j in range(1, self.max_emitterdepth+1): # connect c0 - li - .. - l0
            si_e = EPS[j]
            weight_e = EPW[j]
            active_e_node = EPA[j]
            mis_light = EPM[j]
            
            is_delta_emit = mi.Bool(False)
            if j > 0:
                is_delta_emit = mi.has_flag(si_e.bsdf().flags(), mi.BSDFFlags.Delta)
                
            ds, sensor_val = self.sensor.sample_direction(si_e, sampler.next_2d(), active_e_node)
            active_conn = active_e_node & ~is_delta_emit
             
            shadow_ray = si_e.spawn_ray_to(ds.p)
            active_conn &= ~scene.ray_test(shadow_ray)
            
            d_vec = ds.d
            dist_sq = ds.dist * ds.dist
            
            cos_e = dr.abs(dr.dot(si_e.n, d_vec))
            cos_c = dr.abs(dr.dot(ds.n, -d_vec))
            
            #active_conn &= (cos_e > 0.0) & (cos_c > 0.0)
            
            pdf_sensor_solid = ds.pdf
            pdf_cam_area = dr.select(dist_sq > 1e-6, pdf_sensor_solid * dr.abs(cos_e) / dist_sq, 0.0)
            
            #p_l2c = dr.select(dist_sq > 1e-6, ds.pdf * dr.abs(cos_c) / dist_sq, 0.0)
            
            pdf_light_solid = mi.Float(1.0)
            pdf_light_area = mi.Float(1.0)
            if j == 0:
                #   Emitter 자체인 경우 (Cosine 분포)
                wo_local = si_e.to_local(d_vec)
                pdf_light_solid = mi.warp.square_to_cosine_hemisphere_pdf(wo_local)
                pdf_light_area = dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
                pdf_light_area *= pdf_emit #dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
            else:
                # 표면 반사인 경우 (BSDF 확률)
                wo_local = si_e.to_local(d_vec)
                _, pdf_light_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, wo_local, active_conn)
                pdf_light_area = dr.select(dist_sq > 1e-6, pdf_light_solid * dr.abs(cos_c) / dist_sq, 0.0)
            
            connectpdf = (pdf_cam_area / pdf_light_area)
            #connectpdf *= connectpdf
            
            ratio_cam_vs_light = dr.select(pdf_light_area > 1e-6, mis_light * connectpdf, 0.0)
            
            mis_weight = 1.0 / (1.0 + ratio_cam_vs_light)
            mis_weight = dr.select(active_conn, mis_weight, 0.0)
            mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                     
            L_connect = mi.Spectrum(0.0)
            
            
            if(j == 0):
                si_e.wi = si_e.to_local(wo_local)
                Le = emitter.eval(si_e, active_conn)
                L_connect = weight_e * sensor_val * Le * cos_e
            else:
                G = 1.0 / dist_sq
                bsdf_val = si_e.bsdf().eval(ctx_e, si_e, wo_local, active_conn)
                L_connect = weight_e  * sensor_val * bsdf_val#* bsdf_val
                
            L_connect = dr.select(active_conn, L_connect * mis_weight, 0.0)
            
            
            width_f = mi.Float(self.film_size.x)
            height_f = mi.Float(self.film_size.y)
            
            pos_x = dr.clip(dr.floor(ds.uv.x), 0.0, width_f - 1.0)
            pos_y = dr.clip(dr.floor(ds.uv.y), 0.0, height_f - 1.0)
                
            pixel_idx = (mi.UInt32(pos_y) * self.film_size.x + mi.UInt32(pos_x)) * 3
            

            dr.scatter_add(target=self.light_image_buffer, value=L_connect[0], index=pixel_idx + 0, active=active_conn)
            dr.scatter_add(target=self.light_image_buffer, value=L_connect[1], index=pixel_idx + 1, active=active_conn)
            dr.scatter_add(target=self.light_image_buffer, value=L_connect[2], index=pixel_idx + 2, active=active_conn)
        
        
        beta_c = mi.Spectrum(1.0)
        active_c = active
        mis_cam = mi.Float(0.0)
        prev_pdf_fwd_c = mi.Float(0.0)
        prev_n_c = dr.zeros(mi.Vector3f)
        prev_si = dr.zeros(mi.SurfaceInteraction3f)
        
        prev_bsdf_delta =  mi.Bool(True)
        
        si = scene.ray_intersect(ray)
        Le_direct = si.emitter(scene).eval(si)
        L += beta_c * Le_direct # connect c0 - c1
        
        
        prev_pdf_fwd_c = mi.Float(1.0) #mi.warp.square_to_cosine_hemisphere_pdf(-si.wi)
        
        
        for i in range(self.max_caemradepth):
            si = scene.ray_intersect(ray)
            active_c &= si.is_valid()
            
            bsdf = si.bsdf()
            
            emitter_ptr = si.emitter(scene)
            is_emitt = (emitter_ptr != None)
            is_emitter_hit = active_c & is_emitt
            
            Le_hit = emitter_ptr.eval(si, is_emitter_hit)
            
            if i == 0:
                # Primary Ray (여기서 직접광 계산)
                L += dr.select(is_emitter_hit, beta_c * Le_hit, 0.0)
            else:
                ds_hit = mi.DirectionSample3f(scene, si, prev_si) 
                
                # 2. NEE PDF 계산 (Solid Angle)
                pdf_nee = scene.pdf_emitter_direction(prev_si, ds_hit, ~prev_bsdf_delta)
                
                # Secondary Ray (MIS 적용)
                mis_weight = 1.0 / (1.0 + pdf_nee + (1.0 + mis_cam))
                weight_val = dr.select(is_emitter_hit & dr.isfinite(mis_weight), 
                                     beta_c * Le_hit * mis_weight, 0.0)
                L += weight_val
            
            cos_curr = dr.maximum(0.0, dr.dot(si.sh_frame.n, -ray.d))
            cos_prev = dr.maximum(0.0, dr.dot(prev_n_c, ray.d))
            _, pdf_rev_c = si.bsdf().eval_pdf(ctx_c, si, si.to_local(-ray.d), ~prev_bsdf_delta)
                
                
            denom_c = prev_pdf_fwd_c * cos_curr
            numer_c = pdf_rev_c * cos_prev
            ratio = dr.select(denom_c > 1e-6, numer_c / denom_c, 0.0)
            #ratio *= ratio   
            mis_cam = ratio * (1.0 + mis_cam)
            
            for j in range(self.max_emitterdepth + 1):
                si_e = EPS[j]
                weight_e = EPW[j]
                active_e_node = EPA[j]
                
                active_conn = active_c & active_e_node
                
                d_conn = si_e.p - si.p
                dist_sq = dr.squared_norm(d_conn)
                dist = dr.sqrt(dist_sq)
                wo_c = d_conn / dist
                wo_e = -wo_c
                
                shadow_ray = si.spawn_ray_to(si_e.p)
                
                is_delta_cam = mi.has_flag(bsdf.flags(), mi.BSDFFlags.Delta)
                is_delta_emit = mi.Bool(False)
                if j > 0:
                    is_delta_emit = mi.has_flag(si_e.bsdf().flags(), mi.BSDFFlags.Delta)
                
                visible = active_conn & ~scene.ray_test(shadow_ray) & ~is_delta_cam & ~is_delta_emit
                
                cos_c = dr.dot(si.n, wo_c)
                cos_e = dr.dot(si_e.n, wo_e)
                
                _, pdf_c2l = bsdf.eval_pdf(ctx_c, si, si.to_local(wo_c), visible)
                p_A_c2l = pdf_c2l * dr.abs(cos_e) / dist_sq
                
                p_A_l2c = mi.Float(0.0)
                
                if j == 0:
                    p_A_l2c = pdf_emit #dr.select(weight_e > 1e-6, 1.0 / weight_e, 0.0)
                    pdf_dir = mi.warp.square_to_cosine_hemisphere_pdf(wo_e) * dr.abs(cos_c) / dist_sq
                    p_A_l2c *= pdf_dir
                else:
                    _, pdf_l2c_solid = si_e.bsdf().eval_pdf(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    p_A_l2c = pdf_l2c_solid * dr.abs(cos_c) / dist_sq

                
                cam_side = p_A_c2l / p_A_l2c
                #cam_side *= cam_side
                
                light_side = p_A_l2c / p_A_c2l
                #light_side *= light_side
                
                w_cam_side = dr.select(p_A_l2c > 1e-6, (mis_cam) * (cam_side), 0.0)
                w_light_side = dr.select(p_A_c2l > 1e-6, EPM[j] * (light_side), 0.0)
                
                mis_weight = 1.0 / (1.0 + w_cam_side + w_light_side)
                mis_weight = dr.select(visible, mis_weight, 0.0)
                mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                
                
                contrib = mi.Spectrum(0.0)
                
                if j == 0:
                    si_e.wi = si_e.to_local(wo_e)
                    Le_connect = emitter.eval(si_e, visible)
                    f_cam = bsdf.eval(ctx_c, si, si.to_local(wo_c), visible)
                    G = cos_e / dist_sq
                    contrib = beta_c  * Le_connect * weight_e * G * f_cam
                    
                else:
                    f_cam = bsdf.eval(ctx_c, si, si.to_local(wo_c), visible)
                    f_emit = si_e.bsdf().eval(ctx_e, si_e, si_e.to_local(wo_e), visible)
                    G = 1.0 / dist_sq
                    contrib = beta_c * f_cam * G * f_emit * weight_e 

                mis_weight = dr.select(dr.isfinite(mis_weight), mis_weight, 0.0)
                L += dr.select(visible, contrib*mis_weight, 0.0) # connect c0 - c1 - c(i+1) - l(j) - l(0)


            bsdf_sample, bsdf_weight = bsdf.sample(ctx_c, si, sampler.next_1d(), sampler.next_2d(), active_c)
            prev_pdf_fwd_c = bsdf_sample.pdf
            prev_n_c = si.sh_frame.n
            prev_si = si
            beta_c *= bsdf_weight
            prev_bsdf_delta = mi.has_flag(bsdf_sample.sampled_type, mi.BSDFFlags.Delta)
            ray = si.spawn_ray(si.to_world(bsdf_sample.wo))
            
        
        
        return L, active, [], None